# Grammar–KT pipeline walkthrough

This notebook mirrors `scripts/run.py`: scientific declarations are loaded directly, then passed to chronological stage functions. There is no experiment configuration layer.

`Typed EGP resource → normalisation → canonicalisation → generation → validation → grammar fold → simulation → KC representation → projection → KT → evaluation`

In [1]:
import json, sys
from copy import deepcopy
from functools import partial
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd() if (Path.cwd() / 'modules').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

from grammar_kt.canonicalise import canonicalise
from grammar_kt.evaluate import evaluate
from grammar_kt.fold import apply_fold
from grammar_kt.generate import generate_items
from grammar_kt.io import call_model, load_typed_resource, read_jsonl, read_text, read_yaml
from grammar_kt.kc import project_kcs, select_kcs
from grammar_kt.kt import run_kt
from grammar_kt.normalise import normalise
from grammar_kt.simulate import simulate
from grammar_kt.validate_items import bank_summary, validate_items

LIVE_MODE = False
NOTEBOOK_LEARNERS = 8
NOTEBOOK_PASSES = 2
temporary_run = TemporaryDirectory(prefix='grammar_kt_walkthrough_')
WORK = Path(temporary_run.name)
print('Fixture mode: deterministic model responses' if not LIVE_MODE else 'Live model mode')

## Research declarations

Every file is named and loaded here. The variables below are the actual scientific objects consumed by the stages.

In [2]:
RESOURCE_PATH = ROOT / 'data/fixtures/egp_pilot.jsonl'
resource_schema = read_yaml(ROOT / 'modules/grammar/resource/egp/schema.yaml')

phase1_prompt = read_text(ROOT / 'modules/grammar/resource/egp/normalisation/phase1.txt')
phase2_prompt = read_text(ROOT / 'modules/grammar/resource/egp/normalisation/phase2.txt')
normalisation_rulebook = read_text(ROOT / 'modules/grammar/resource/egp/normalisation/rulebook.md')
grammar_schema = read_yaml(ROOT / 'modules/grammar/canonical/schema.yaml')

generation_prompt = read_text(ROOT / 'modules/items/generation/prompt.txt')
generation_rulebook = read_text(ROOT / 'modules/items/generation/rulebook.md')
generation_design = read_yaml(ROOT / 'modules/items/generation/design.yaml')
item_format = read_yaml(ROOT / 'modules/items/generation/formats/controlled_production.yaml')
lexicon = read_jsonl(ROOT / 'modules/items/generation/lexicon.jsonl')

validation_prompt = read_text(ROOT / 'modules/items/validation/prompt.txt')
validation_criteria = read_yaml(ROOT / 'modules/items/validation/criteria.yaml')

grammar_fold_spec = read_yaml(ROOT / 'modules/simulation/folds/reference.yaml')
simulation_world = read_yaml(ROOT / 'modules/simulation/world.yaml')

kc_policy = read_yaml(ROOT / 'modules/kcs/policies/factorized.yaml')
kc_candidate_space = read_yaml(ROOT / 'modules/kcs/candidates.yaml')
kc_obligations = read_yaml(ROOT / 'modules/kcs/obligations.yaml')
kc_selector = read_yaml(ROOT / 'modules/kcs/selector.yaml')

kt_protocol = read_yaml(ROOT / 'modules/evaluation/kt/protocol.yaml')
evaluation_protocol = read_yaml(ROOT / 'modules/evaluation/protocol.yaml')

if LIVE_MODE:
    model_call = call_model
    normalisation_model = generation_model = 'gpt-5.6-sol'
    validation_model = 'gpt-5.6-terra'
    reasoning_effort = 'medium'
else:
    model_call = partial(
        call_model,
        fixture_responses=read_yaml(ROOT / 'data/fixtures/model_responses.yaml'),
    )
    normalisation_model = generation_model = validation_model = 'fixture'
    reasoning_effort = 'deterministic'

display(pd.DataFrame([
    {'stage': 'resource', 'declaration': resource_schema['resource_id']},
    {'stage': 'normalisation', 'declaration': phase1_prompt.splitlines()[0] + ' / ' + phase2_prompt.splitlines()[0]},
    {'stage': 'canonicalisation', 'declaration': grammar_schema['schema_id']},
    {'stage': 'generation', 'declaration': generation_design['design_id'] + ' / ' + item_format['format_id']},
    {'stage': 'validation', 'declaration': validation_criteria['policy_id']},
    {'stage': 'grammar fold', 'declaration': grammar_fold_spec['fold_id']},
    {'stage': 'simulation', 'declaration': simulation_world['world_id']},
    {'stage': 'KC representation', 'declaration': kc_policy['policy_id']},
    {'stage': 'KT', 'declaration': kt_protocol['protocol_id']},
    {'stage': 'evaluation', 'declaration': evaluation_protocol['protocol_id']},
]))

## Typed EGP resource → normalisation

In [3]:
resources = load_typed_resource(RESOURCE_PATH, resource_schema)
display(pd.DataFrame(resources)[['source_id', 'subcategory', 'guideword', 'cefr']])

mappings = normalise(
    resources,
    phase1_prompt,
    phase2_prompt,
    normalisation_rulebook,
    grammar_schema,
    model=normalisation_model,
    reasoning_effort=reasoning_effort,
    model_call=model_call,
    evidence_dir=WORK / 'normalisation',
)
display(pd.DataFrame([{
    'source_id': row['source_id'], 'result': row['result'],
    'cells': len(row['cells']), 'note': row['note'],
} for row in mappings]))
evidence = WORK / 'normalisation/calls/egp_present_simple_phase1/rendered_prompt.txt'
display(Markdown('**Rendered Phase-1 prompt excerpt**'))
print(evidence.read_text()[:700])

## Canonicalisation → item generation

In [4]:
cells = canonicalise(mappings, grammar_schema)
display(pd.DataFrame([{'cell_id': cell['cell_id'], **cell['features']} for cell in cells]))

display(Markdown('**Generation declarations**'))
display(pd.DataFrame(lexicon[:3]))
print(generation_design)
print(item_format)
print(generation_rulebook.splitlines()[0])

candidates = generate_items(
    cells,
    generation_prompt,
    generation_rulebook,
    generation_design,
    item_format,
    lexicon,
    model=generation_model,
    reasoning_effort=reasoning_effort,
    model_call=model_call,
    evidence_dir=WORK / 'generation',
)
display(pd.DataFrame(candidates)[['item_id', 'cell_id', 'prompt', 'target_answer']])

## Independent validation → fixed item bank

In [5]:
display(pd.DataFrame([
    {'criterion': name, **declaration}
    for name, declaration in validation_criteria['criteria'].items()
]))
accepted_items, judgments = validate_items(
    candidates,
    cells,
    validation_prompt,
    validation_criteria,
    model=validation_model,
    reasoning_effort=reasoning_effort,
    model_call=model_call,
    evidence_dir=WORK / 'validation',
)
item_bank_summary = bank_summary(candidates, accepted_items, judgments, cells)
display(pd.DataFrame(judgments)[['item_id', 'accepted']])
print({key: item_bank_summary[key] for key in ('accepted_items', 'acceptance_rate', 'covered_cells')})

## Grammar fold → ontology-independent simulation

The notebook reduces only the learner count and number of passes so the complete walkthrough stays quick.

In [6]:
grammar_fold = apply_fold(cells, grammar_fold_spec)
display(pd.DataFrame(grammar_fold)[['cell_id', 'grammar_split']])

small_world = deepcopy(simulation_world)
small_world['learners'] = NOTEBOOK_LEARNERS
small_world['passes'] = NOTEBOOK_PASSES
events = simulate(
    accepted_items,
    grammar_fold,
    small_world,
    oracle_path=WORK / 'oracle_debug.json',
)
display(pd.DataFrame(events).head(8))
print({'learners': len({row['learner_id'] for row in events}), 'events': len(events), 'seed': small_world['seed']})

## KC representation → mechanical projection

The baseline uses the declared factorized policy directly. The selected-policy call separately demonstrates development-only selection from its three scientific declarations.

In [7]:
policy = kc_policy
selected_policy = select_kcs(
    cells,
    accepted_items,
    grammar_fold,
    kc_candidate_space,
    kc_obligations,
    kc_selector,
)
display(pd.DataFrame(policy['kcs'])[['id', 'definition']])
print('Selected from development:', [row['id'] for row in selected_policy['kcs']])

projection = project_kcs(accepted_items, cells, policy)
display(pd.DataFrame(projection))

## Knowledge tracing → evaluation

In [8]:
predictions = run_kt(events, projection, kt_protocol)
display(pd.DataFrame(predictions).head(9))

results = evaluate(
    candidates,
    judgments,
    accepted_items,
    cells,
    grammar_fold,
    events,
    policy,
    projection,
    predictions,
    evaluation_protocol,
)
display(pd.DataFrame([
    {'technique': name, **{key: metrics[key] for key in ('n', 'log_loss', 'brier_score', 'auc')}}
    for name, metrics in results['kt'].items()
]))

## Executed object flow

In [9]:
WALKTHROUGH_SUMMARY = {
    'live_mode': LIVE_MODE,
    'source_descriptors': len(resources),
    'mappings': len(mappings),
    'canonical_cells': len(cells),
    'candidate_items': len(candidates),
    'accepted_items': len(accepted_items),
    'learners': len({event['learner_id'] for event in events}),
    'events': len(events),
    'baseline_kcs': len(policy['kcs']),
    'kt_techniques': kt_protocol['techniques'],
}
display(pd.DataFrame([
    ('Typed resource', len(resources)),
    ('Normalised mappings', len(mappings)),
    ('Canonical cells', len(cells)),
    ('Candidate items', len(candidates)),
    ('Accepted items', len(accepted_items)),
    ('Learner events', len(events)),
    ('KC projections', len(projection)),
    ('KT predictions', len(predictions)),
], columns=['scientific object', 'records']))
print(json.dumps(WALKTHROUGH_SUMMARY, indent=2))